# Mission 3 P0 재실행 (보험 제출 로그)

- 모델: TF-IDF + OvR Logistic Regression (`scripts/mission3_baseline 최종 수정본.py`)
- 분할: Training 서울 JSON **80:20 multilabel stratified**, seed=42, threshold=0.5
- 판정: 내부 Val Macro F1 **0.5676 ±0.003** (로컬 기준 0.56755) → 범위 안이면 이 실행의 joblib + `pip_freeze.txt`를 M3 보험 제출본으로 확정
- 코드: main `4bfd2c4` zip · CPU 런타임 · **셀 출력 지우지 말 것**
- 실행 방식: Colab 계정 동시 세션 1개 제한 때문에, M2 L2 스펙트로그램 빌드가 돌고 있는 같은 CPU VM에서 `jupyter nbconvert --execute`로 실행했다(셀 출력 보존). 원본 대비 변경은 0번 셀의 drive.mount → 마운트 확인, 코드 zip 자동 압축 해제뿐이다.


In [ ]:
# 0. 설정 (실행 담당이 채움)
# 단일 세션 제한으로 M2 스펙트로그램 빌드 중인 같은 Colab VM에서 nbconvert로 실행 (Drive는 이미 마운트됨)
import os
assert os.path.isdir('/content/drive/MyDrive'), 'Drive not mounted'

import os, sys, subprocess, glob, hashlib, json
from pathlib import Path

REPO = Path('/content/dcc2026-prelim-main-4bfd2c4')   # 4bfd2c4 zip 압축 푼 폴더
DATA_ROOT = Path('/content/drive/.shortcut-targets-by-id/1Ezl13Cpmt2lGgl-FqISYJWfop8omii6z/대학부 데이터')
LABEL_DIR = None   # None이면 아래에서 Training 서울 라벨 폴더 자동 탐색
OUT = Path('/content/drive/MyDrive/dcc_runs/m3_p0_rerun')
OUT.mkdir(parents=True, exist_ok=True)

SCRIPT = REPO / 'scripts' / 'mission3_baseline 최종 수정본.py'   # 파일명에 공백 있음
REF_MACRO_F1, TOL = 0.5676, 0.003
if not SCRIPT.is_file():
    subprocess.run(['unzip','-q','-o','/content/drive/MyDrive/dcc2026-prelim-main-4bfd2c4.zip','-d','/content/'], check=True)
if not SCRIPT.is_file():  # zip 최상위 폴더명이 다를 때 자동 탐색
    hits = glob.glob('/content/*/scripts/mission3_baseline 최종 수정본.py') + glob.glob('/content/scripts/mission3_baseline 최종 수정본.py')
    print('script hits:', hits)
    if hits:
        SCRIPT = Path(hits[0]); REPO = SCRIPT.parent.parent
print('REPO =', REPO, '| exists:', REPO.is_dir())
assert SCRIPT.is_file(), SCRIPT

In [ ]:
# 1. 의존성 + 라벨 폴더 확인
!pip install -q iterative-stratification

if LABEL_DIR is None:
    cands = [Path(p) for p in glob.glob(str(DATA_ROOT / '**' / 'TL_서울_구급'), recursive=True)
             if 'Training' in p or '학습' in p]
    print('후보:', *cands, sep='\n  ')
    assert len(cands) == 1, '후보가 1개가 아니면 LABEL_DIR을 직접 지정'
    LABEL_DIR = cands[0]
n_json = sum(1 for _ in Path(LABEL_DIR).rglob('*.json'))
print('LABEL_DIR =', LABEL_DIR)
print('Training JSON =', n_json, '(기대 29,200)')
assert n_json == 29200

In [ ]:
# 2. 실시간 출력 run() (팀 공통 규칙)
def run(cmd):
    print('$', ' '.join(map(str, cmd)), flush=True)
    with open(OUT / 'run_log.txt', 'a', encoding='utf-8') as log:
        p = subprocess.Popen([sys.executable, '-u', *map(str, cmd)], cwd=REPO,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             env={**os.environ, 'PYTHONUNBUFFERED': '1'})
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
    assert p.wait() == 0, f'failed: {cmd}'

In [ ]:
# 3. 학습 + 내부 Val (80:20 strat, seed 42, th 0.5)
run([SCRIPT, '--label_dir', LABEL_DIR, '--output_dir', OUT,
     '--test_size', '0.2', '--seed', '42', '--threshold', '0.5'])

In [ ]:
# 4. 판정 + ckpt SHA
m = json.loads((OUT / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(m, ensure_ascii=False, indent=2))
import pandas as pd
print(pd.read_csv(OUT / 'per_label_metrics.csv').to_string(index=False))

ckpt = OUT / 'mission3_tfidf_logreg.joblib'
sha = hashlib.sha256(ckpt.read_bytes()).hexdigest()
diff = m['macro_f1'] - REF_MACRO_F1
ok = abs(diff) <= TOL and m['total_sample_count'] == 29200 and m['load_failure_count'] == 0
verdict = {'macro_f1': m['macro_f1'], 'ref': REF_MACRO_F1, 'diff': diff, 'tol': TOL,
           'pass': ok, 'ckpt': str(ckpt), 'ckpt_sha256': sha, 'repo_commit': '4bfd2c4'}
(OUT / 'verdict.json').write_text(json.dumps(verdict, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(verdict, ensure_ascii=False, indent=2))
print('재현 성공 → 보험 제출본 확정 후보' if ok else '범위 이탈 → 채택 보류, 원인 확인')

In [ ]:
# 5. 패키지 버전 고정
!pip freeze > "{OUT}/pip_freeze.txt"
!grep -iE "scikit-learn|numpy|pandas|scipy|joblib|iterative" "{OUT}/pip_freeze.txt"